<h1><b>Restricted Boltzmann Machine and Deep Belief Network</b></h1>

<p align="justify">In this exercise, you will examine how <i>RBM (<a href="https://en.wikipedia.org/wiki/Restricted_Boltzmann_machine">Restricted Boltzmann Machine</a>)</i> and <i>DBNs (<a href="https://en.wikipedia.org/wiki/Deep_belief_network">Deep Belief Networks</a>)</i> work, using the program provided below. This program utilizes the <a href="https://en.wikipedia.org/wiki/MNIST_database"><i>MNIST</i> dataset</a>, which is a large database of handwritten digits commonly used for training various image processing systems. For this exercise, you will use the file <i>mnist_original.mat</i>, which is available <a href="https://www.kaggle.com/datasets/avnishnish/mnist-original?resource=download">here</a>.</p>

<p align="justify">One known application of <i>RBMs</i> is feature extraction (feature representation) from a dataset, with the goal of representing the input (visible neurons) by a lower-dimensional vector (hidden neurons). In this exercise, you will compare the accuracy of a digit classifier based on the <i>Logistic Regression</i> algorithm when it takes as input: (i) the dataset without any preprocessing by the <i>RBM</i>, (ii) the dataset after it has been processed by the <i>RBM</i>, and (iii) the dataset processed using a <i>DBN</i>, which is two stacked <i>RBMs</i>.</p>

<p align="justify">Based on the code provided, answer the following questions:</p> <ul> <li>Briefly describe how <i>RBM</i> works. What are its differences compared with a <i>Boltzmann Machine</i>?</li> <li>What is the idea behind <i>DBNs</i>, and in which cases can be used?</li> <li>Mention the main applications of <i>RBMs</i> and <i>DBNs</i>.</li><li>Compare the classification results obtained with the <i>Logistic Regression</i> algorithm without using an <i>RBM</i>, with the results obtained when an <i>RBM</i> is used, as well as when <i>RBMs</i> and <i>DBNs</i> are used for feature extraction. What do you observe regarding the accuracy of the results?</li> </ul>

In [ ]:
#!/usr/bin/env python
# source: https://stackoverflow.com/questions/52166308/stacking-rbms-to-create-deep-belief-network-in-sklearn

import numpy as np
import matplotlib.pyplot as plt

from scipy.io import loadmat
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import BernoulliRBM
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

def norm(arr):
    arr = arr.astype(float)
    arr -= arr.min()
    arr /= arr.max()
    return arr

if __name__ == '__main__':

    # load MNIST data set
    mnist = loadmat("mnist-original.mat")
    X, Y = mnist["data"].T, mnist["label"][0]

    # normalize inputs to 0-1 range
    X = norm(X)

    # split into train, validation, and test data sets
    X_train, X_test, Y_train, Y_test = train_test_split(X,       Y,       test_size=10000, random_state=0)
    X_train, X_val,  Y_train, Y_val  = train_test_split(X_train, Y_train, test_size=10000, random_state=0)

    # --------------------------------------------------------------------------------
    # set hyperparameters

    learning_rate = 0.02
    total_units   =  800
    total_epochs  =   50
    batch_size    =  128

    C = 100. # optimum for benchmark model according to sklearn docs: https://scikit-learn.org/stable/auto_examples/neural_networks/plot_rbm_logistic_classification.html#sphx-glr-auto-examples-neural-networks-plot-rbm-logistic-classification-py)

    # --------------------------------------------------------------------------------
    # construct models

    # RBM
    rbm = BernoulliRBM(n_components=total_units, learning_rate=learning_rate, batch_size=batch_size, n_iter=total_epochs, verbose=1)

    # "output layer"
    logistic = LogisticRegression(C=C, solver='lbfgs', multi_class='multinomial', max_iter=200, verbose=1)

    models = []
    models.append(Pipeline(steps=[('logistic', clone(logistic))]))                                              # base model / benchmark
    models.append(Pipeline(steps=[('rbm1', clone(rbm)), ('logistic', clone(logistic))]))                        # single RBM
    models.append(Pipeline(steps=[('rbm1', clone(rbm)), ('rbm2', clone(rbm)), ('logistic', clone(logistic))]))  # RBM stack / DBN

    # --------------------------------------------------------------------------------
    # train and evaluate models

    for model in models:
        # train
        model.fit(X_train, Y_train)

        # evaluate using validation set
        print("Model performance:\n%s\n" % (
            classification_report(Y_val, model.predict(X_val))))